In [1]:
# ================================
# 1️⃣ Install dependencies
# ================================
!pip install -qU "google-genai>=1.0.0" chromadb pypdf

In [2]:
from pypdf import PdfReader
import chromadb
from google import genai
from google.colab import userdata
import os

In [3]:
# Initialize the Gemini client
GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

In [7]:
# ================================
# 3️⃣ Load PDF from Colab directory
# ================================

# Directory containing PDF documents
pdf_directory = "/content/sample_data/"

raw_text = ""

# Iterate through all PDF files in the directory
for filename in os.listdir(pdf_directory):
    if filename.endswith(".pdf"):
        pdf_path = os.path.join(pdf_directory, filename)
        reader = PdfReader(pdf_path)

        for page in reader.pages:
            text = page.extract_text()
            if text:
                raw_text += text + "\n"

# Raise an error if no PDFs were found
if not raw_text:
    raise ValueError("No PDF files found in the specified directory. Please upload your PDFs or update `pdf_directory`.")

print("PDFs loaded. Preview:")
#print(raw_text[:1000])  # first 1000 chars


ValueError: No PDF files found in the specified directory. Please upload your PDFs or update `pdf_directory`.

In [5]:
# ================================
# 4️⃣ Chunk the document
# ================================
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text)
print("Total chunks created:", len(chunks))
#print("\n--- Content of Each Chunk ---")
#for i, chunk in enumerate(chunks):
#   print(f"Chunk {i+1}:\n{chunk}\n--------------------")
#print("----------------------------")

Total chunks created: 0


In [6]:
# ================================
# 6️⃣ Store chunks in ChromaDB
# ================================
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="simple_rag")

collection.add(
    documents=chunks,
    ids=[f"id_{i}" for i in range(len(chunks))]
)

print("Documents added to ChromaDB.")

# /root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz added to chroma db


ValueError: Non-empty lists are required for ['ids', 'documents'] in add.

In [ ]:
import google.generativeai as genai

# --- 4. Simple RAG Query ---
while True:
    user_query = input("Ask a question (type 'exit' to quit): ")

    if user_query.lower() == 'exit':
        print("Exiting Q&A session.")
        break

    # Retrieve top 2 matches
    results = collection.query(query_texts=[user_query], n_results=2)
    context = " ".join(results['documents'][0])

    print(f"\nRetrieved chunks content:\n{results['documents'][0]}") # Added line to print chunks content

    # Generate Final Answer
    prompt = f"Context: {context}\n\nQuestion: {user_query}\n\nAnswer:"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt)

    # Correctly display the number of retrieved chunks (n_results)
    print(f"Retrieved {len(results['documents'][0])} chunks from the knowledge base.")
    print(f"\nAnswer: {response.text}")